## ML flow setup

First, we want to create a uv environment using:
```bash
uv init
uv venv
```

then install dependencies with
```bash
uv add mlflow torch torchvision
```

Note - you may want to link to a specific cuda installation for pytorch depending on your system, if so, [copy the relevent code in this page to your pyproject.toml first](https://docs.astral.sh/uv/guides/integration/pytorch/#using-a-pytorch-index) then run the above command.

then we create a jupyter kernel with
```bash
uv add --dev ipykernel
uv run ipython kernel install --user --env VIRTUAL_ENV=$(pwd)/.venv --name=project 
```

Then we simply create a jupyter notebook and select our newly created kernel as its kernel and write the below code

When we want to launch the mlflow server, we run this command
```bash
docker compose up
```


In [24]:
import mlflow

# The set_experiment API creates a new experiment if it doesn't exist.
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Deep Learning Experiment")

# IMPORTANT: Enable system metrics monitoring
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(1)

In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# Load and prepare data
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.FashionMNIST("data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST("data", train=False, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)

cuda


In [26]:
import torch.nn as nn

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork().to(device)

In [27]:
params = {
    "epochs": 5,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "optimizer": "SGD",
    "model_type": "MLP",
    "hidden_units": [512, 512, 512],
}

# Define optimizer and loss function
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=params["learning_rate"])

In [28]:
with mlflow.start_run() as run:
    # Log training parameters
    mlflow.log_params(params)

    for epoch in range(params["epochs"]):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            # Forward pass
            optimizer.zero_grad()
            output = model(data)
            loss = loss_fn(output, target)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Calculate metrics
            train_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            # Log batch metrics (every 100 batches)
            if batch_idx % 100 == 0:
                batch_loss = train_loss / (batch_idx + 1)
                batch_acc = 100.0 * correct / total
                mlflow.log_metrics(
                    {"batch_loss": batch_loss, "batch_accuracy": batch_acc},
                    step=epoch * len(train_loader) + batch_idx,
                )

        # Calculate epoch metrics
        epoch_loss = train_loss / len(train_loader)
        epoch_acc = 100.0 * correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = loss_fn(output, target)

                val_loss += loss.item()
                _, predicted = output.max(1)
                val_total += target.size(0)
                val_correct += predicted.eq(target).sum().item()

        # Calculate and log epoch validation metrics
        val_loss = val_loss / len(test_loader)
        val_acc = 100.0 * val_correct / val_total

        # Log epoch metrics
        mlflow.log_metrics(
            {
                "train_loss": epoch_loss,
                "train_accuracy": epoch_acc,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
            },
            step=epoch,
        )
        # Log checkpoint at the end of each epoch
        mlflow.pytorch.log_model(model, name=f"checkpoint_{epoch}")

        print(
            f"Epoch {epoch + 1}/{params['epochs']}, "
            f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%"
        )

    # Log the final trained model
    model_info = mlflow.pytorch.log_model(model, name="final_model")

2026/06/08 04:17:46 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/06/08 04:17:46 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/06/08 04:18:03 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\uvtest
2026/06/08 04:18:03 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/08 04:18:03 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it i

Epoch 1/5, Train Loss: 2.1867, Train Acc: 38.62%, Val Loss: 2.0108, Val Acc: 55.58%


2026/06/08 04:18:19 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\uvtest
2026/06/08 04:18:20 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/08 04:18:20 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/08 04:18:20 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\uvtest
202

Epoch 2/5, Train Loss: 1.6571, Train Acc: 57.19%, Val Loss: 1.3173, Val Acc: 61.46%


2026/06/08 04:18:37 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\uvtest
2026/06/08 04:18:38 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/08 04:18:38 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/08 04:18:38 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\uvtest
202

...

Epoch 4/5, Train Loss: 0.9054, Train Acc: 68.69%, Val Loss: 0.8597, Val Acc: 69.05%


2026/06/08 04:19:12 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\uvtest
2026/06/08 04:19:13 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/08 04:19:13 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/08 04:19:13 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\uvtest
202

Epoch 5/5, Train Loss: 0.8027, Train Acc: 71.55%, Val Loss: 0.7800, Val Acc: 71.68%


2026/06/08 04:19:14 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/08 04:19:14 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/08 04:19:14 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\uvtest
2026/06/08 04:19:14 INFO mlflow.utils.environment: Detected uv project at c:\uvtest. Attempting to export requirements

🏃 View run gaudy-eel-682 at: http://localhost:5000/#/experiments/1/runs/bfa4f778d1d44a14bb55cbef4d358c3a
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [29]:
# Load the final model
model = mlflow.pytorch.load_model("runs:/bfa4f778d1d44a14bb55cbef4d358c3a/final_model")

model.to(device)
model.eval()

# Resume the previous run to log test metrics
with mlflow.start_run(run_id=run.info.run_id) as run:
    # Evaluate the model on the test set
    test_loss, test_correct, test_total = 0, 0, 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = loss_fn(output, target)

            test_loss += loss.item()
            _, predicted = output.max(1)
            test_total += target.size(0)
            test_correct += predicted.eq(target).sum().item()

    # Calculate and log final test metrics
    test_loss = test_loss / len(test_loader)
    test_acc = 100.0 * test_correct / test_total

    mlflow.log_metrics({"test_loss": test_loss, "test_accuracy": test_acc})
    print(f"Final Test Accuracy: {test_acc:.2f}%")

2026/06/08 04:21:33 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/06/08 04:21:33 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/06/08 04:21:35 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/06/08 04:21:35 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Final Test Accuracy: 71.68%
🏃 View run gaudy-eel-682 at: http://localhost:5000/#/experiments/1/runs/bfa4f778d1d44a14bb55cbef4d358c3a
🧪 View experiment at: http://localhost:5000/#/experiments/1
